# Асинхронный веб-краулер

### Установка зависимостей

In [1]:
%pip install aiohttp aiofiles beautifulsoup4 lxml

Note: you may need to restart the kernel to use updated packages.


### Импорты
Все библиотеки, которые используются в ноутбуке, собраны здесь.

In [2]:
import asyncio
import aiohttp
import logging
import time
import json
import itertools
import re
import aiofiles
import random

from collections import Counter, deque
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse, urldefrag
from contextlib import asynccontextmanager, suppress
from urllib.robotparser import RobotFileParser


### Настройка логирования

In [3]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("crawler")

## Класс HTMLParser

In [4]:
class HTMLParser:
    def empty_result(self, url: str) -> dict:
        return {
            "url": url,
            "title": "",
            "text": "",
            "links": [],
            "metadata": {},
            "images": [],
            "headings": {},
            "tables": [],
            "lists": [],
        }

    async def parse_html(self, html: str, url: str) -> dict:
        result = self.empty_result(url)
        try:
            soup = await asyncio.to_thread(BeautifulSoup, html, "lxml")
        except Exception as e:
            logger.warning("Не удалось разобрать HTML (%s): %s", type(e).__name__, url)
            return result

        extractors = {
            "metadata": lambda: self.extract_metadata(soup),
            "text": lambda: self.extract_text(soup),
            "links": lambda: self.extract_links(soup, url),
            "images": lambda: self.extract_images(soup, url),
            "headings": lambda: self.extract_headings(soup),
            "tables": lambda: self.extract_tables(soup),
            "lists": lambda: self.extract_lists(soup),
        }
        for field, extract in extractors.items():
            try:
                result[field] = extract()
            except Exception as e:
                logger.warning(
                    "Ошибка при извлечении %s (%s): %s", field, type(e).__name__, url
                )

        result["title"] = result["metadata"].get("title", "")
        return result

    def extract_links(self, soup: BeautifulSoup, base_url: str) -> list[str]:
        links = []
        for tag in soup.find_all("a", href=True):
            absolute_url = urljoin(base_url, tag["href"])
            if self.is_valid_url(absolute_url):
                links.append(absolute_url)
        return links

    def is_valid_url(self, url: str) -> bool:
        parsed = urlparse(url)
        return parsed.scheme in ("http", "https") and bool(parsed.netloc)

    def extract_text(self, soup: BeautifulSoup, selector: str = None) -> str:
        root = soup.select_one(selector) if selector else soup
        if root is None:
            return ""
        return root.get_text(separator=" ", strip=True)

    def extract_metadata(self, soup: BeautifulSoup) -> dict:
        def meta_content(name: str) -> str:
            tag = soup.find("meta", attrs={"name": name})
            return tag.get("content", "") if tag else ""

        title = soup.title.get_text(strip=True) if soup.title else ""
        return {
            "title": title,
            "description": meta_content("description"),
            "keywords": meta_content("keywords"),
        }
    
    def extract_images(self, soup: BeautifulSoup, base_url: str) -> list[dict]:
        images = []
        for tag in soup.find_all("img", src=True):
            images.append({
                "src": urljoin(base_url, tag["src"]),
                "alt": tag.get("alt", ""),
            })
        return images

    def extract_headings(self, soup: BeautifulSoup) -> dict[str, list[str]]:
        headings = {}
        for level in ("h1", "h2", "h3"):
            headings[level] = [
                tag.get_text(strip=True) for tag in soup.find_all(level)
            ]
        return headings

    def extract_tables(self, soup: BeautifulSoup) -> list[list[list[str]]]:
        tables = []
        for table in soup.find_all("table"):
            rows = []
            for tr in table.find_all("tr"):
                cells = [
                    cell.get_text(strip=True) for cell in tr.find_all(["th", "td"])
                ]
                if cells:
                    rows.append(cells)
            tables.append(rows)
        return tables

    def extract_lists(self, soup: BeautifulSoup) -> list[dict]:
        lists = []
        for tag in soup.find_all(["ul", "ol"]):
            items = [
                li.get_text(strip=True) for li in tag.find_all("li", recursive=False)
            ]
            lists.append({"type": tag.name, "items": items})
        return lists

## Класс CrawlerQueue

In [5]:
class CrawlerQueue:
    def __init__(self):
        self.queue = asyncio.PriorityQueue()
        self.counter = itertools.count()
        self.seen: set[str] = set()
        self.processed: set[str] = set()
        self.failed: dict[str, str] = {}

    def add_url(self, url: str, priority: int = 0):
        if url in self.seen:
            return
        self.seen.add(url)
        self.queue.put_nowait((-priority, next(self.counter), url))

    def requeue(self, url: str, priority: int = 0):
        self.queue.put_nowait((-priority, next(self.counter), url))

    async def get_next(self) -> str | None:
        if self.queue.empty():
            return None
        _, _, url = await self.queue.get()
        return url

    def mark_processed(self, url: str):
        self.processed.add(url)

    def mark_failed(self, url: str, error: str):
        self.failed[url] = error

    def get_stats(self) -> dict:
        return {
            "queued": self.queue.qsize(),
            "processed": len(self.processed),
            "failed": len(self.failed),
            "total_seen": len(self.seen),
        }

## Класс SemaphoreManager

In [6]:
class SemaphoreManager:
    def __init__(self, global_limit: int = 10, per_domain_limit: int = 2):
        self.global_semaphore = asyncio.Semaphore(global_limit)
        self.per_domain_limit = per_domain_limit
        self.domain_semaphores: dict[str, asyncio.Semaphore] = {}
        self.active_total = 0
        self.active_by_domain: dict[str, int] = {}

    @asynccontextmanager
    async def acquire(self, url: str):
        domain = urlparse(url).netloc
        if domain not in self.domain_semaphores:
            self.domain_semaphores[domain] = asyncio.Semaphore(self.per_domain_limit)

        async with self.domain_semaphores[domain]:
            async with self.global_semaphore:
                self.active_total += 1
                self.active_by_domain[domain] = self.active_by_domain.get(domain, 0) + 1
                try:
                    yield
                finally:
                    self.active_total -= 1
                    self.active_by_domain[domain] -= 1

    def get_active(self) -> dict:
        return {
            "total": self.active_total,
            "by_domain": {
                domain: count
                for domain, count in self.active_by_domain.items()
                if count > 0
            },
        }

## Класс RateLimiter

In [7]:
class RateLimiter:
    def __init__(
        self,
        requests_per_second: float = 1.0,
        per_domain: bool = True,
        min_delay: float = 0.0,
        jitter: float = 0.0,
        stats_window: float = 5.0,
    ):
        self.interval = max(1.0 / requests_per_second, min_delay)
        self.jitter = jitter
        self.per_domain = per_domain
        self.next_allowed: dict[str, float] = {}
        self.domain_intervals: dict[str, float] = {}

        self.stats_window = stats_window
        self.recent_requests: deque[float] = deque()
        self.first_request: float | None = None
        self.last_request: dict[str, float] = {}
        self.waiting_domains: set[str] = set()
        self.gaps_total = 0.0
        self.gaps_count = 0
        self.min_gap: float | None = None

    def set_domain_delay(self, domain: str, delay: float):
        self.domain_intervals[domain] = max(self.interval, delay)

    def time_until_allowed(self, domain: str = None) -> float:
        key = domain if self.per_domain and domain else "global"
        return max(0.0, self.next_allowed.get(key, 0.0) - time.monotonic())

    def mark_waiting(self, domain: str):
        self.waiting_domains.add(domain)

    async def acquire(self, domain: str = None):
        key = domain if self.per_domain and domain else "global"
        interval = self.domain_intervals.get(key, self.interval)
        now = time.monotonic()
        slot = max(now, self.next_allowed.get(key, now))
        self.next_allowed[key] = slot + interval + random.uniform(0, self.jitter)
        wait = slot - now
        if wait > 0:
            await asyncio.sleep(wait)
        self.record_request(domain or "global", throttled=wait > 0)

    def record_request(self, domain: str, throttled: bool):
        now = time.monotonic()
        if self.first_request is None:
            self.first_request = now
        self.recent_requests.append(now)
        throttled = throttled or domain in self.waiting_domains
        self.waiting_domains.discard(domain)
        if throttled and domain in self.last_request:
            gap = now - self.last_request[domain]
            self.gaps_total += gap
            self.gaps_count += 1
            if self.min_gap is None or gap < self.min_gap:
                self.min_gap = gap
        self.last_request[domain] = now

    def get_stats(self) -> dict:
        now = time.monotonic()
        while self.recent_requests and now - self.recent_requests[0] > self.stats_window:
            self.recent_requests.popleft()
        if self.first_request is None:
            current_rate = 0.0
        else:
            window = max(min(self.stats_window, now - self.first_request), 1.0)
            current_rate = len(self.recent_requests) / window
        avg_delay = self.gaps_total / self.gaps_count if self.gaps_count else 0.0
        return {
            "requests_per_second": current_rate,
            "avg_delay": avg_delay,
            "min_delay": self.min_gap or 0.0,
        }

## Класс RobotsParser

In [8]:
class RobotsParser:
    def __init__(self, session: aiohttp.ClientSession):
        self.session = session
        self.rules: dict[str, RobotFileParser] = {}
        self.crawl_delays: dict[str, dict[str, float]] = {}
        self.locks: dict[str, asyncio.Lock] = {}

    async def fetch_robots(self, base_url: str) -> dict:
        parsed = urlparse(base_url)
        domain = parsed.netloc
        robots_url = f"{parsed.scheme}://{domain}/robots.txt"

        if domain not in self.locks:
            self.locks[domain] = asyncio.Lock()
        async with self.locks[domain]:
            if domain not in self.rules:
                parser = RobotFileParser(robots_url)
                delays: dict[str, float] = {}
                try:
                    async with self.session.get(robots_url) as response:
                        if response.status == 200:
                            text = await response.text()
                            parser.parse(text.splitlines())
                            delays = self.parse_crawl_delays(text)
                        elif response.status in (401, 403):
                            parser.disallow_all = True
                        else:
                            parser.allow_all = True
                except (aiohttp.ClientError, asyncio.TimeoutError) as e:
                    logger.warning("Не удалось загрузить robots.txt (%s): %s", type(e).__name__, robots_url)
                    parser.allow_all = True
                self.rules[domain] = parser
                self.crawl_delays[domain] = delays

        parser = self.rules[domain]
        return {
            "domain": domain,
            "robots_url": robots_url,
            "allow_all": parser.allow_all,
            "disallow_all": parser.disallow_all,
            "crawl_delay": self.get_crawl_delay(domain=domain),
        }

    def parse_crawl_delays(self, text: str) -> dict[str, float]:
        delays: dict[str, float] = {}
        agents: list[str] = []
        in_rules = False
        for raw_line in text.splitlines():
            line = raw_line.split("#", 1)[0].strip()
            if ":" not in line:
                continue
            field, value = line.split(":", 1)
            field = field.strip().lower()
            value = value.strip()
            if field == "user-agent":
                if in_rules:
                    agents = []
                    in_rules = False
                agents.append(value.lower())
                continue
            in_rules = True
            if field == "crawl-delay":
                try:
                    delay = float(value)
                except ValueError:
                    continue
                for agent in agents:
                    delays[agent] = delay
        return delays

    def can_fetch(self, url: str, user_agent: str = "*") -> bool:
        parser = self.rules.get(urlparse(url).netloc)
        if parser is None:
            return True
        return parser.can_fetch(user_agent, url)

    def get_crawl_delay(self, user_agent: str = "*", domain: str | None = None) -> float:
        delays = self.crawl_delays.get(domain, {})
        name = user_agent.split("/")[0].lower()
        for agent, delay in delays.items():
            if agent != "*" and agent in name:
                return delay
        return delays.get("*", 0.0)

## Класс AsyncCrawler

In [9]:
class AsyncCrawler:
    def __init__(
        self,
        max_concurrent: int = 10,
        max_depth: int = 2,
        per_domain_limit: int | None = None,
        requests_per_second: float | None = None,
        respect_robots: bool = False,
        user_agent: str = "AsyncCrawler/1.0",
        min_delay: float = 0.0,
        jitter: float = 0.0,
        max_retries: int = 0,
        backoff_base: float = 1.0,
        connect_timeout: float = 5,
        read_timeout: float = 10,
    ):
        self.max_concurrent = max_concurrent
        self.max_depth = max_depth
        self.respect_robots = respect_robots
        self.user_agent = user_agent
        self.jitter = jitter
        self.max_retries = max_retries
        self.backoff_base = backoff_base
        self.semaphores = SemaphoreManager(
            global_limit=max_concurrent,
            per_domain_limit=per_domain_limit or max_concurrent,
        )
        self.parser = HTMLParser()
        self.rate_limiter = RateLimiter(
            requests_per_second=requests_per_second or float("inf"),
            min_delay=min_delay,
            jitter=jitter,
        )

        self.queue = CrawlerQueue()
        self.visited_urls: set[str] = set()
        self.failed_urls: dict[str, str] = {}
        self.processed_urls: dict[str, dict] = {}
        self.blocked_urls: set[str] = set()
        self.url_depths: dict[str, int] = {}
        self.in_progress = 0
        self.same_domain_only = False
        self.allowed_domains: set[str] = set()
        self.exclude_patterns: list[re.Pattern] = []
        self.include_patterns: list[re.Pattern] = []

        timeout = aiohttp.ClientTimeout(
            connect=connect_timeout,
            sock_read=read_timeout,
        )
        # используется системный поиск адресов, чтобы краулер работал через VPN
        connector = aiohttp.TCPConnector(
            limit=max_concurrent,
            resolver=aiohttp.ThreadedResolver(),
        )

        self.session = aiohttp.ClientSession(
            timeout=timeout,
            connector=connector,
            headers={"User-Agent": user_agent},
        )

        self.robots = RobotsParser(self.session)

    async def check_robots(self, url: str) -> bool:
        domain = urlparse(url).netloc
        await self.robots.fetch_robots(url)
        delay = self.robots.get_crawl_delay(self.user_agent, domain)
        if delay:
            self.rate_limiter.set_domain_delay(domain, delay)
        if self.robots.can_fetch(url, self.user_agent):
            return True
        logger.warning("Заблокировано robots.txt: %s", url)
        self.blocked_urls.add(url)
        return False

    async def fetch_url(self, url: str) -> str | None:
        if self.respect_robots and not await self.check_robots(url):
            return None

        error = None
        for attempt in range(self.max_retries + 1):
            if attempt > 0:
                delay = self.backoff_base * 2 ** (attempt - 1) + random.uniform(0, self.jitter)
                logger.warning(
                    "Повтор %d из %d через %.1f сек: %s", attempt, self.max_retries, delay, url
                )
                await asyncio.sleep(delay)
            text, error, retryable = await self.try_fetch(url)
            if text is not None:
                return text
            if not retryable:
                break

        self.failed_urls[url] = error
        return None

    async def try_fetch(self, url: str) -> tuple[str | None, str | None, bool]:
        async with self.semaphores.acquire(url):
            await self.rate_limiter.acquire(urlparse(url).netloc)
            logger.info("Начало загрузки: %s", url)
            try:
                async with self.session.get(url) as response:
                    response.raise_for_status()
                    text = await response.text()
                    logger.info(
                        "Успешно: %s (статус %s, %d символов)",
                        url, response.status, len(text),
                    )
                    return text, None, False
            except aiohttp.ClientResponseError as e:
                logger.error("%s, статус %s: %s", type(e).__name__, e.status, url)
                retryable = e.status >= 500 or e.status == 429
                return None, f"{type(e).__name__} {e.status}", retryable
            except asyncio.TimeoutError as e:
                logger.error("%s, таймаут: %s", type(e).__name__, url)
                return None, type(e).__name__, True
            except aiohttp.ClientError as e:
                logger.error("%s, сетевая ошибка: %s", type(e).__name__, url)
                return None, type(e).__name__, True

    async def fetch_urls(self, urls: list[str]) -> dict[str, str]:
        tasks = [self.fetch_url(url) for url in urls]
        pages = await asyncio.gather(*tasks)
        return {url: page for url, page in zip(urls, pages) if page is not None}

    async def fetch_and_parse(self, url: str) -> dict:
        html = await self.fetch_url(url)
        if html is None:
            return self.parser.empty_result(url)
        return await self.parser.parse_html(html, url)

    async def crawl(
        self,
        start_urls: list[str],
        max_pages: int = 100,
        same_domain_only: bool = False,
        exclude_patterns: list[str] | None = None,
        include_patterns: list[str] | None = None,
    ) -> dict[str, dict]:
        self.queue = CrawlerQueue()
        self.visited_urls = set()
        self.failed_urls = {}
        self.processed_urls = {}
        self.blocked_urls = set()
        self.url_depths = {}
        self.in_progress = 0

        self.same_domain_only = same_domain_only
        self.allowed_domains = {urlparse(url).netloc for url in start_urls}
        self.exclude_patterns = [re.compile(p) for p in exclude_patterns or []]
        self.include_patterns = [re.compile(p) for p in include_patterns or []]

        for url in start_urls:
            url, _ = urldefrag(url)
            self.url_depths[url] = 0
            self.queue.add_url(url, priority=0)

        start_time = time.perf_counter()
        progress = asyncio.create_task(self.report_progress(start_time))
        workers = [
            asyncio.create_task(self.worker(max_pages))
            for _ in range(self.max_concurrent)
        ]
        try:
            await asyncio.gather(*workers)
        finally:
            progress.cancel()
            with suppress(asyncio.CancelledError):
                await progress
            self.print_progress(start_time)
        return self.processed_urls

    def is_allowed(self, url: str) -> bool:
        if self.same_domain_only and urlparse(url).netloc not in self.allowed_domains:
            return False
        if any(pattern.search(url) for pattern in self.exclude_patterns):
            return False
        if self.include_patterns and not any(
            pattern.search(url) for pattern in self.include_patterns
        ):
            return False
        return True

    async def report_progress(self, start_time: float, interval: float = 1.0):
        while True:
            await asyncio.sleep(interval)
            self.print_progress(start_time)

    def print_progress(self, start_time: float):
        stats = self.queue.get_stats()
        rate_stats = self.rate_limiter.get_stats()
        elapsed = time.perf_counter() - start_time
        speed = stats["processed"] / elapsed if elapsed > 0 else 0
        print(
            f"Обработано: {stats['processed']} | "
            f"В очереди: {stats['queued']} | "
            f"Ошибок: {stats['failed']} | "
            f"Заблокировано: {len(self.blocked_urls)} | "
            f"Скорость: {speed:.1f} стр/сек | "
            f"Запросов: {rate_stats['requests_per_second']:.1f} req/sec | "
            f"Средняя задержка: {rate_stats['avg_delay']:.2f} сек"
        )

    async def worker(self, max_pages: int):
        while len(self.visited_urls) < max_pages:
            url = await self.queue.get_next()
            if url is None:
                if self.in_progress == 0:
                    break
                await asyncio.sleep(0.1)
                continue

            domain = urlparse(url).netloc
            wait = self.rate_limiter.time_until_allowed(domain)
            if wait > 0:
                self.rate_limiter.mark_waiting(domain)
                self.queue.requeue(url, priority=-self.url_depths[url])
                await asyncio.sleep(min(wait, 0.05))
                continue

            self.visited_urls.add(url)
            self.in_progress += 1
            try:
                await self.process_url(url)
            finally:
                self.in_progress -= 1

    async def process_url(self, url: str):
        page = await self.fetch_and_parse(url)

        if url in self.blocked_urls:
            return
        if url in self.failed_urls:
            self.queue.mark_failed(url, self.failed_urls[url])
            return

        self.processed_urls[url] = page
        self.queue.mark_processed(url)
        self.expand_links(url)

    def expand_links(self, url: str):
        depth = self.url_depths[url]
        if depth >= self.max_depth:
            return
        for link in self.processed_urls[url]["links"]:
            link, _ = urldefrag(link)
            if self.is_allowed(link):
                self.add_link(link, depth + 1)

    def add_link(self, link: str, depth: int):
        known_depth = self.url_depths.get(link)
        if known_depth is not None and known_depth <= depth:
            return
        self.url_depths[link] = depth
        if link in self.processed_urls:
            self.expand_links(link)
        else:
            self.queue.add_url(link, priority=-depth)

    async def close(self):
        await self.session.close()

## Демонстрация 1 дня

In [10]:
DEMO_URLS = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/get",
    "https://httpbin.org/html",
    "https://httpbin.org/delay/1",
    "https://httpbin.org/delay/2",
    "https://httpbin.org/delay/3",
    "https://httpbin.org/status/404",
]


async def run_sequential(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    results = {}
    start = time.perf_counter()
    try:
        for url in urls:
            page = await crawler.fetch_url(url)
            if page is not None:
                results[url] = page
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


async def run_parallel(urls: list[str]) -> tuple[dict[str, str], float]:
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        results = await crawler.fetch_urls(urls)
    finally:
        await crawler.close()
    return results, time.perf_counter() - start


def print_report(title: str, urls: list[str], results: dict[str, str], elapsed: float):
    print(f"\n{title}")
    for url in urls:
        mark = "✅" if url in results else "❌"
        print(f"  {mark} {url}")
    print(f"  Загружено {len(results)} из {len(urls)} за {elapsed:.2f} сек")


async def main():
    seq_results, seq_time = await run_sequential(DEMO_URLS)
    par_results, par_time = await run_parallel(DEMO_URLS)

    print_report("Последовательная загрузка", DEMO_URLS, seq_results, seq_time)
    print_report("Параллельная загрузка", DEMO_URLS, par_results, par_time)
    print(f"\nПараллельно быстрее в {seq_time / par_time:.1f} раза")


await main()

19:08:08 [INFO] Начало загрузки: https://example.com
19:08:08 [INFO] Успешно: https://example.com (статус 200, 577 символов)
19:08:08 [INFO] Начало загрузки: https://www.python.org
19:08:09 [INFO] Успешно: https://www.python.org (статус 200, 51928 символов)
19:08:09 [INFO] Начало загрузки: https://httpbin.org/get
19:08:10 [INFO] Успешно: https://httpbin.org/get (статус 200, 312 символов)
19:08:10 [INFO] Начало загрузки: https://httpbin.org/html
19:08:10 [INFO] Успешно: https://httpbin.org/html (статус 200, 3739 символов)
19:08:10 [INFO] Начало загрузки: https://httpbin.org/delay/1
19:08:11 [INFO] Успешно: https://httpbin.org/delay/1 (статус 200, 362 символов)
19:08:11 [INFO] Начало загрузки: https://httpbin.org/delay/2
19:08:13 [INFO] Успешно: https://httpbin.org/delay/2 (статус 200, 362 символов)
19:08:13 [INFO] Начало загрузки: https://httpbin.org/delay/3
19:08:17 [INFO] Успешно: https://httpbin.org/delay/3 (статус 200, 362 символов)
19:08:17 [INFO] Начало загрузки: https://httpbin.o


Последовательная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 9.01 сек

Параллельная загрузка
  ✅ https://example.com
  ✅ https://www.python.org
  ✅ https://httpbin.org/get
  ✅ https://httpbin.org/html
  ✅ https://httpbin.org/delay/1
  ✅ https://httpbin.org/delay/2
  ✅ https://httpbin.org/delay/3
  ❌ https://httpbin.org/status/404
  Загружено 7 из 8 за 4.19 сек

Параллельно быстрее в 2.1 раза


## Тестирование 1 дня

In [11]:
async def test_valid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://example.com",
            "https://httpbin.org/html",
        ])
    finally:
        await crawler.close()
    assert len(results) == 2, f"ожидали 2 страницы, получили {len(results)}"
    assert "Example Domain" in results["https://example.com"], "не тот текст страницы"


async def test_invalid_urls():
    crawler = AsyncCrawler()
    try:
        results = await crawler.fetch_urls([
            "https://this-site-does-not-exist-12345.com",
            "https://httpbin.org/status/404",
        ])
    finally:
        await crawler.close()
    assert results == {}, f"ожидали пустой результат, получили {list(results)}"


async def test_timeout():
    crawler = AsyncCrawler(read_timeout=1)
    start = time.perf_counter()
    try:
        page = await crawler.fetch_url("https://httpbin.org/delay/5")
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start
    assert page is None, "страница не должна была загрузиться"
    assert elapsed < 4, f"таймаут не сработал, ждали {elapsed:.1f} сек"


async def test_parallel_faster():
    urls = [
        "https://httpbin.org/delay/1",
        "https://httpbin.org/delay/2",
        "https://httpbin.org/delay/3",
    ]
    _, seq_time = await run_sequential(urls)
    _, par_time = await run_parallel(urls)
    assert par_time < seq_time * 0.75, (
        f"параллельно {par_time:.1f} сек, последовательно {seq_time:.1f} сек"
    )


async def run_tests(tests: list):
    logger.setLevel(logging.WARNING)
    passed = 0
    try:
        for test in tests:
            try:
                await test()
                print(f"✅ {test.__name__}")
                passed += 1
            except AssertionError as e:
                print(f"❌ {test.__name__}: {e}")
    finally:
        logger.setLevel(logging.INFO)
    print(f"\nПройдено {passed} из {len(tests)}")


await run_tests([test_valid_urls, test_invalid_urls, test_timeout, test_parallel_faster])

✅ test_valid_urls


19:08:22 [ERROR] ClientConnectorDNSError, сетевая ошибка: https://this-site-does-not-exist-12345.com
19:08:23 [ERROR] ClientResponseError, статус 404: https://httpbin.org/status/404


✅ test_invalid_urls


19:08:24 [ERROR] SocketTimeoutError, таймаут: https://httpbin.org/delay/5


✅ test_timeout
✅ test_parallel_faster

Пройдено 4 из 4


## Демонстрация 2 дня

In [12]:
DEMO_PAGES = [
    "https://example.com",
    "https://www.python.org",
    "https://httpbin.org/html",
    "https://docs.python.org/3/",
]


def summarize(page: dict) -> dict:
    return {
        "url": page["url"],
        "title": page["title"],
        "h1": page["headings"].get("h1", []),
        "text_length": len(page["text"]),
        "links_count": len(page["links"]),
        "links": page["links"][:5],
        "images_count": len(page["images"]),
    }


async def demo_parsing(urls: list[str]):
    crawler = AsyncCrawler(max_concurrent=5)
    start = time.perf_counter()
    try:
        pages = await asyncio.gather(*(crawler.fetch_and_parse(url) for url in urls))
    finally:
        await crawler.close()
    elapsed = time.perf_counter() - start

    for page in pages:
        print(json.dumps(summarize(page), ensure_ascii=False, indent=2))

    total_links = sum(len(page["links"]) for page in pages)
    total_text = sum(len(page["text"]) for page in pages)
    print("\nСтатистика")
    print(f"  Страниц: {len(pages)}")
    print(f"  Всего ссылок: {total_links}")
    print(f"  Общая длина текста: {total_text} символов")
    print(f"  Время: {elapsed:.2f} сек")


await demo_parsing(DEMO_PAGES)

19:08:36 [INFO] Начало загрузки: https://example.com
19:08:36 [INFO] Начало загрузки: https://www.python.org
19:08:36 [INFO] Начало загрузки: https://httpbin.org/html
19:08:36 [INFO] Начало загрузки: https://docs.python.org/3/
19:08:36 [INFO] Успешно: https://www.python.org (статус 200, 51928 символов)
19:08:36 [INFO] Успешно: https://docs.python.org/3/ (статус 200, 19620 символов)
19:08:36 [INFO] Успешно: https://example.com (статус 200, 577 символов)
19:08:37 [INFO] Успешно: https://httpbin.org/html (статус 200, 3739 символов)


{
  "url": "https://example.com",
  "title": "Example Domain",
  "h1": [],
  "text_length": 171,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://www.python.org",
  "title": "Welcome to Python.org",
  "h1": [
    "Functions Defined",
    "All the Flow You’d Expect",
    "Compound Data Types",
    "Intuitive Interpretation",
    "Quick & Easy to Learn"
  ],
  "text_length": 6657,
  "links_count": 212,
  "links": [
    "https://www.python.org#content",
    "https://www.python.org#python-network",
    "https://www.python.org/",
    "https://www.python.org/psf/",
    "https://docs.python.org"
  ],
  "images_count": 1
}
{
  "url": "https://httpbin.org/html",
  "title": "",
  "h1": [
    "Herman Melville - Moby-Dick"
  ],
  "text_length": 3594,
  "links_count": 0,
  "links": [],
  "images_count": 0
}
{
  "url": "https://docs.python.org/3/",
  "title": "3.15.0 Documentation",
  "h1": [
    "Python 3.15.0 documentation"
  ],
  "text_length": 3217,
  "links_count": 9

## Тестирование 2 дня

In [13]:
VALID_HTML = """
<html>
  <head>
    <title>Тестовая страница</title>
    <meta name="description" content="Описание страницы">
  </head>
  <body>
    <h1>Главный заголовок</h1>
    <p>Основной текст страницы.</p>
    <img src="/logo.png" alt="Логотип">
    <table><tr><th>Город</th></tr><tr><td>Москва</td></tr></table>
    <ul><li>Чай</li><li>Кофе</li></ul>
  </body>
</html>
"""


async def test_parse_valid_html():
    result = await HTMLParser().parse_html(VALID_HTML, "https://example.com")
    assert result["title"] == "Тестовая страница", f"title: {result['title']}"
    assert result["metadata"]["description"] == "Описание страницы", "нет description"
    assert result["headings"]["h1"] == ["Главный заголовок"], "неверный h1"
    assert "Основной текст страницы." in result["text"], "нет текста"
    assert result["images"] == [
        {"src": "https://example.com/logo.png", "alt": "Логотип"}
    ], "неверные картинки"
    assert result["tables"] == [[["Город"], ["Москва"]]], "неверная таблица"
    assert result["lists"] == [{"type": "ul", "items": ["Чай", "Кофе"]}], "неверный список"


async def test_parse_broken_html():
    parser = HTMLParser()
    broken = "<title>Битая</title><h1>Заголовок<p>Текст<a href='/page'>ссылка<table><tr><td>A<td>B"
    result = await parser.parse_html(broken, "https://example.com")
    assert result["title"] == "Битая", f"title: {result['title']}"
    assert result["links"] == ["https://example.com/page"], f"links: {result['links']}"
    assert result["tables"] == [[["A", "B"]]], f"tables: {result['tables']}"

    result = await parser.parse_html(None, "https://example.com")
    assert result == parser.empty_result("https://example.com"), "ожидали пустой результат"


async def test_extract_links():
    html = """
    <a href="https://python.org">1</a>
    <a href="mailto:info@example.com">2</a>
    <a href="javascript:void(0)">3</a>
    <a href="tel:+70000000000">4</a>
    <a>без href</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com")
    assert links == ["https://python.org"], f"links: {links}"


async def test_relative_urls():
    html = """
    <a href="/about">1</a>
    <a href="page2.html">2</a>
    <a href="../archive">3</a>
    <a href="//cdn.example.org/lib">4</a>
    """
    soup = BeautifulSoup(html, "lxml")
    links = HTMLParser().extract_links(soup, "https://example.com/news/today")
    assert links == [
        "https://example.com/about",
        "https://example.com/news/page2.html",
        "https://example.com/archive",
        "https://cdn.example.org/lib",
    ], f"links: {links}"


await run_tests([test_parse_valid_html, test_parse_broken_html, test_extract_links, test_relative_urls])

19:08:37 [WARNING] Не удалось разобрать HTML (TypeError): https://example.com


✅ test_parse_valid_html
✅ test_parse_broken_html
✅ test_extract_links
✅ test_relative_urls

Пройдено 4 из 4


## Демонстрация 3 дня

In [14]:
async def save_results(crawler: AsyncCrawler, path: str):
    data = {
        "processed": [
            {**page, "depth": crawler.url_depths[url]}
            for url, page in crawler.processed_urls.items()
        ],
        "failed": crawler.failed_urls,
    }
    async with aiofiles.open(path, "w", encoding="utf-8") as file:
        await file.write(json.dumps(data, ensure_ascii=False, indent=2))


async def demo_crawl():
    crawler = AsyncCrawler(max_concurrent=10, max_depth=2, per_domain_limit=3)
    logger.setLevel(logging.WARNING)
    try:
        results = await crawler.crawl(
            start_urls=[
                "https://quotes.toscrape.com/",
                "https://books.toscrape.com/",
            ],
            max_pages=150,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()

    await save_results(crawler, "crawl_results.json")

    by_domain = Counter(urlparse(url).netloc for url in results)
    by_depth = Counter(crawler.url_depths[url] for url in results)
    print(f"\nОбработано: {len(results)} страниц")
    print(f"Ошибок: {len(crawler.failed_urls)}")
    print(f"По сайтам: {dict(by_domain)}")
    print(f"По глубине: {dict(sorted(by_depth.items()))}")
    print("Результаты сохранены в crawl_results.json")


await demo_crawl()

Обработано: 2 | В очереди: 109 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.0 стр/сек | Запросов: 5.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 6 | В очереди: 107 | Ошибок: 0 | Заблокировано: 0 | Скорость: 3.0 стр/сек | Запросов: 4.5 req/sec | Средняя задержка: 0.00 сек
Обработано: 14 | В очереди: 135 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.7 стр/сек | Запросов: 5.7 req/sec | Средняя задержка: 0.00 сек
Обработано: 24 | В очереди: 157 | Ошибок: 0 | Заблокировано: 0 | Скорость: 6.0 стр/сек | Запросов: 6.7 req/sec | Средняя задержка: 0.00 сек
Обработано: 29 | В очереди: 152 | Ошибок: 0 | Заблокировано: 0 | Скорость: 5.8 стр/сек | Запросов: 6.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 39 | В очереди: 164 | Ошибок: 0 | Заблокировано: 0 | Скорость: 6.5 стр/сек | Запросов: 7.6 req/sec | Средняя задержка: 0.00 сек
Обработано: 53 | В очереди: 211 | Ошибок: 0 | Заблокировано: 0 | Скорость: 7.6 стр/сек | Запросов: 9.8 req/sec | Средняя задержка: 0.00 сек
Обработано: 65 | В оче

## Тестирование 3 дня

In [15]:
FAKE_SITE = {
    "https://site.com/": '<a href="/a">a</a><a href="/b">b</a><a href="https://other.com/x">x</a>',
    "https://site.com/a": '<a href="/a/deep">deep</a><a href="/">home</a><a href="/b">b</a>',
    "https://site.com/b": '<a href="/a">a</a><a href="/login">login</a>',
    "https://site.com/a/deep": '<a href="/a/deeper">deeper</a>',
    "https://site.com/a/deeper": "",
    "https://site.com/login": "",
    "https://other.com/x": "",
}


class FakeCrawler(AsyncCrawler):
    def __init__(self, site: dict | None = None, delays: dict | None = None, **kwargs):
        super().__init__(**kwargs)
        self.site = site or FAKE_SITE
        self.delays = delays or {}
        self.fetch_calls: list[str] = []

    async def fetch_url(self, url: str) -> str | None:
        self.fetch_calls.append(url)
        await asyncio.sleep(self.delays.get(url, 0.01))
        if url in self.site:
            return f"<html><body>{self.site[url]}</body></html>"
        self.failed_urls[url] = "NotFound"
        return None


async def run_fake_crawl(
    max_depth: int, site: dict | None = None, delays: dict | None = None, **filters
) -> FakeCrawler:
    crawler = FakeCrawler(site=site, delays=delays, max_concurrent=3, max_depth=max_depth)
    try:
        await crawler.crawl(["https://site.com/"], max_pages=100, **filters)
    finally:
        await crawler.close()
    return crawler


async def test_queue_priority():
    queue = CrawlerQueue()
    queue.add_url("https://site.com/a", priority=0)
    queue.add_url("https://site.com/b", priority=5)
    queue.add_url("https://site.com/c", priority=0)
    queue.add_url("https://site.com/d", priority=-1)
    queue.add_url("https://site.com/b", priority=10)

    order = [await queue.get_next() for _ in range(5)]
    assert order == [
        "https://site.com/b",
        "https://site.com/a",
        "https://site.com/c",
        "https://site.com/d",
        None,
    ], f"порядок: {order}"


async def test_depth_limit():
    crawler = await run_fake_crawl(max_depth=1)
    depths = [crawler.url_depths[url] for url in crawler.visited_urls]
    assert max(depths) == 1, f"глубины: {depths}"
    assert "https://site.com/a/deep" not in crawler.visited_urls, "зашли глубже max_depth"

    crawler = await run_fake_crawl(max_depth=2)
    assert "https://site.com/a/deep" in crawler.visited_urls, "не дошли до глубины 2"
    assert "https://site.com/a/deeper" not in crawler.visited_urls, "зашли на глубину 3"


async def test_url_filters():
    crawler = await run_fake_crawl(
        max_depth=3, same_domain_only=True, exclude_patterns=[r"/login"]
    )
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/b",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"

    crawler = await run_fake_crawl(max_depth=3, include_patterns=[r"/a"])
    assert crawler.visited_urls == {
        "https://site.com/",
        "https://site.com/a",
        "https://site.com/a/deep",
        "https://site.com/a/deeper",
    }, f"visited: {crawler.visited_urls}"


async def test_no_duplicates():
    crawler = await run_fake_crawl(max_depth=3)
    calls = crawler.fetch_calls
    assert len(calls) == len(set(calls)), f"повторные загрузки: {calls}"
    assert set(calls) == crawler.visited_urls, "visited_urls не совпадает с загрузками"
    assert len(crawler.visited_urls) == len(FAKE_SITE), f"посещено {len(crawler.visited_urls)}"

SHORTCUT_SITE = {
    "https://site.com/": '<a href="/slow">slow</a><a href="/fast1">fast1</a>',
    "https://site.com/slow": '<a href="/target">target</a>',
    "https://site.com/fast1": '<a href="/fast2">fast2</a>',
    "https://site.com/fast2": '<a href="/target">target</a>',
    "https://site.com/target": '<a href="/child">child</a>',
    "https://site.com/child": "",
}

async def test_shortest_depth():
    crawler = await run_fake_crawl(
        max_depth=3,
        site=SHORTCUT_SITE,
        delays={"https://site.com/slow": 0.5},
    )
    assert crawler.url_depths["https://site.com/target"] == 2, (
        f"глубина target: {crawler.url_depths['https://site.com/target']}"
    )
    assert "https://site.com/child" in crawler.visited_urls, "страница в пределах max_depth потеряна"

await run_tests([test_queue_priority, test_depth_limit, test_url_filters, test_no_duplicates, test_shortest_depth])


✅ test_queue_priority
Обработано: 4 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 39.5 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 6 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 59.2 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_depth_limit
Обработано: 5 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 49.3 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 4 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 39.5 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_url_filters
Обработано: 7 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 69.3 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_no_duplicates
Обработано: 6 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 9.9 стр/сек | Запросов: 0.0 req/sec | Средняя задержка: 0.00 сек
✅ test_shortest_depth

Пройдено 5 из 5


## Демонстрация 4 дня

In [16]:
async def demo_polite_crawl():
    crawler = AsyncCrawler(
        max_concurrent=5,
        max_depth=1,
        requests_per_second=2.0,
        respect_robots=True,
        min_delay=0.5,
        jitter=0.2,
        user_agent="MyBot/1.0",
    )
    logger.setLevel(logging.WARNING)
    start = time.perf_counter()
    try:
        results = await crawler.crawl(
            start_urls=[
                "https://docs.python.org/3/",
                "https://quotes.toscrape.com/",
            ],
            max_pages=50,
            same_domain_only=True,
        )
    finally:
        logger.setLevel(logging.INFO)
        await crawler.close()
    elapsed = time.perf_counter() - start

    rate_stats = crawler.rate_limiter.get_stats()
    by_domain = Counter(urlparse(url).netloc for url in results)
    print("\nИтоги")
    print(f"  Обработано страниц: {len(results)}")
    print(f"  По сайтам: {dict(by_domain)}")
    print(f"  Заблокировано robots.txt: {len(crawler.blocked_urls)}")
    print(f"  Ошибок: {len(crawler.failed_urls)}")
    print(f"  Время: {elapsed:.1f} сек")

    print("\nЗадержки")
    print(f"  Разрешено не чаще чем раз в: {crawler.rate_limiter.interval:.2f} сек (плюс до {crawler.jitter:.2f} случайно)")
    print(f"  Самая короткая фактическая пауза: {rate_stats['min_delay']:.2f} сек")
    print(f"  Средняя фактическая пауза: {rate_stats['avg_delay']:.2f} сек")

    print("\nЗаблокированные адреса")
    for url in sorted(crawler.blocked_urls):
        print(f"  {url}")


await demo_polite_crawl()

Обработано: 1 | В очереди: 44 | Ошибок: 0 | Заблокировано: 0 | Скорость: 1.0 стр/сек | Запросов: 2.0 req/sec | Средняя задержка: 0.00 сек
Обработано: 4 | В очереди: 87 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.0 стр/сек | Запросов: 3.3 req/sec | Средняя задержка: 0.75 сек
Обработано: 5 | В очереди: 84 | Ошибок: 0 | Заблокировано: 0 | Скорость: 1.7 стр/сек | Запросов: 3.2 req/sec | Средняя задержка: 0.81 сек
Обработано: 8 | В очереди: 82 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.0 стр/сек | Запросов: 2.8 req/sec | Средняя задержка: 0.81 сек
Обработано: 10 | В очереди: 81 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.0 стр/сек | Запросов: 2.4 req/sec | Средняя задержка: 0.82 сек
Обработано: 13 | В очереди: 79 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.2 стр/сек | Запросов: 2.2 req/sec | Средняя задержка: 0.87 сек
Обработано: 15 | В очереди: 76 | Ошибок: 0 | Заблокировано: 0 | Скорость: 2.1 стр/сек | Запросов: 2.2 req/sec | Средняя задержка: 0.88 сек
Обработано: 18 | В очереди: 74 

19:09:10 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.4/
19:09:10 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.3/
19:09:10 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.2/
19:09:10 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.1/
19:09:10 [WARNING] Заблокировано robots.txt: https://docs.python.org/3.0/
19:09:10 [WARNING] Заблокировано robots.txt: https://docs.python.org/2.7/
19:09:10 [WARNING] Заблокировано robots.txt: https://docs.python.org/2.6/


Обработано: 42 | В очереди: 42 | Ошибок: 0 | Заблокировано: 7 | Скорость: 2.5 стр/сек | Запросов: 3.0 req/sec | Средняя задержка: 0.79 сек
Обработано: 43 | В очереди: 42 | Ошибок: 0 | Заблокировано: 7 | Скорость: 2.5 стр/сек | Запросов: 3.0 req/sec | Средняя задержка: 0.79 сек

Итоги
  Обработано страниц: 43
  По сайтам: {'docs.python.org': 21, 'quotes.toscrape.com': 22}
  Заблокировано robots.txt: 7
  Ошибок: 0
  Время: 17.1 сек

Задержки
  Разрешено не чаще чем раз в: 0.50 сек (плюс до 0.20 случайно)
  Самая короткая фактическая пауза: 0.54 сек
  Средняя фактическая пауза: 0.79 сек

Заблокированные адреса
  https://docs.python.org/2.6/
  https://docs.python.org/2.7/
  https://docs.python.org/3.0/
  https://docs.python.org/3.1/
  https://docs.python.org/3.2/
  https://docs.python.org/3.3/
  https://docs.python.org/3.4/


## Тестирование 4 дня

In [17]:
class FakeResponse:
    def __init__(self, status: int, text: str):
        self.status = status
        self._text = text

    async def text(self) -> str:
        return self._text

    async def __aenter__(self):
        return self

    async def __aexit__(self, *args):
        return False


class FakeSession:
    def __init__(self, pages: dict[str, str]):
        self.pages = pages
        self.calls: list[str] = []

    def get(self, url: str) -> FakeResponse:
        self.calls.append(url)
        if url in self.pages:
            return FakeResponse(200, self.pages[url])
        return FakeResponse(404, "")


class FakePoliteCrawler(AsyncCrawler):
    def __init__(self, site: dict[str, str], robots_pages: dict[str, str], **kwargs):
        super().__init__(**kwargs)
        self.site = site
        self.robots = RobotsParser(FakeSession(robots_pages))
        self.request_times: list[float] = []

    async def try_fetch(self, url: str) -> tuple[str | None, str | None, bool]:
        async with self.semaphores.acquire(url):
            await self.rate_limiter.acquire(urlparse(url).netloc)
            self.request_times.append(time.monotonic())
            if url in self.site:
                return f"<html><body>{self.site[url]}</body></html>", None, False
            return None, "NotFound", False


def gaps_between(times: list[float]) -> list[float]:
    times = sorted(times)
    return [later - earlier for earlier, later in zip(times, times[1:])]


async def test_rate_limit_single_domain():
    limiter = RateLimiter(requests_per_second=10.0)
    times = []

    async def request():
        await limiter.acquire("a.com")
        times.append(time.monotonic())

    await asyncio.gather(*(request() for _ in range(5)))
    gaps = gaps_between(times)
    assert all(gap >= 0.09 for gap in gaps), f"паузы: {gaps}"


async def test_rate_limit_multiple_domains():
    domains = ["a.com", "a.com", "a.com", "b.com", "b.com", "b.com"]
    for per_domain, expected in ((True, 0.2), (False, 0.5)):
        limiter = RateLimiter(requests_per_second=10.0, per_domain=per_domain)
        start = time.monotonic()
        await asyncio.gather(*(limiter.acquire(domain) for domain in domains))
        elapsed = time.monotonic() - start
        assert expected - 0.02 <= elapsed <= expected + 0.05, (
            f"per_domain={per_domain}: {elapsed:.2f} сек вместо {expected}"
        )


ROBOTS_TXT = """
User-agent: *
Disallow: /private/
Crawl-delay: 2

User-agent: BadBot
Disallow: /

User-agent: SlowBot
Crawl-delay: 0.5
"""


async def test_robots_parsing():
    session = FakeSession({"https://site.com/robots.txt": ROBOTS_TXT})
    robots = RobotsParser(session)
    await robots.fetch_robots("https://site.com/page")
    info = await robots.fetch_robots("https://site.com/other")
    assert session.calls == ["https://site.com/robots.txt"], f"загрузок: {session.calls}"
    assert info["crawl_delay"] == 2.0, f"crawl_delay: {info['crawl_delay']}"
    assert robots.get_crawl_delay("SlowBot/1.0", "site.com") == 0.5, "дробный Crawl-delay не прочитан"
    assert robots.can_fetch("https://site.com/public") is True, "public должен быть разрешён"
    assert robots.can_fetch("https://site.com/private/data") is False, "private должен быть запрещён"
    assert robots.can_fetch("https://site.com/public", "BadBot") is False, "BadBot должен быть заблокирован"

    info = await robots.fetch_robots("https://empty.com/")
    assert info["allow_all"] is True, "без robots.txt всё разрешено"


POLITE_SITE = {
    "https://polite.com/": '<a href="/public">public</a><a href="/private/secret">secret</a>',
    "https://polite.com/public": "",
    "https://polite.com/private/secret": "",
}


async def test_blocked_urls():
    crawler = FakePoliteCrawler(
        site=POLITE_SITE,
        robots_pages={"https://polite.com/robots.txt": "User-agent: *\nDisallow: /private/\n"},
        respect_robots=True,
        max_depth=1,
    )
    try:
        results = await crawler.crawl(["https://polite.com/"])
    finally:
        await crawler.close()
    assert crawler.blocked_urls == {"https://polite.com/private/secret"}, f"blocked: {crawler.blocked_urls}"
    assert set(results) == {"https://polite.com/", "https://polite.com/public"}, f"results: {set(results)}"
    assert len(crawler.request_times) == 2, f"запросов: {len(crawler.request_times)}"


async def test_delays_respected():
    limiter = RateLimiter(requests_per_second=100.0, min_delay=0.2, jitter=0.1)
    times = []
    for _ in range(4):
        await limiter.acquire("a.com")
        times.append(time.monotonic())
    gaps = gaps_between(times)
    assert all(0.19 <= gap <= 0.32 for gap in gaps), f"min_delay и jitter: {gaps}"

    site = {f"https://polite.com/page{i}": "" for i in range(4)}
    crawler = FakePoliteCrawler(
        site=site,
        robots_pages={"https://polite.com/robots.txt": "User-agent: *\nCrawl-delay: 0.5\n"},
        respect_robots=True,
    )
    try:
        await crawler.fetch_urls(list(site))
    finally:
        await crawler.close()
    gaps = gaps_between(crawler.request_times)
    assert all(gap >= 0.49 for gap in gaps), f"Crawl-delay: {gaps}"


def make_site(domain: str, pages: int) -> dict[str, str]:
    links = "".join(f'<a href="/page{i}">{i}</a>' for i in range(pages))
    site = {f"https://{domain}/": links}
    site.update({f"https://{domain}/page{i}": "" for i in range(pages)})
    return site


async def test_domains_in_parallel():
    site = {**make_site("a.com", 6), **make_site("b.com", 6)}
    crawler = FakePoliteCrawler(
        site=site,
        robots_pages={},
        requests_per_second=2.0,
        max_concurrent=3,
        max_depth=1,
    )
    start = time.monotonic()
    try:
        results = await crawler.crawl(["https://a.com/", "https://b.com/"])
    finally:
        await crawler.close()
    elapsed = time.monotonic() - start
    assert len(results) == 14, f"обработано {len(results)}"
    assert elapsed < 4.0, f"сайты обходились по очереди: {elapsed:.1f} сек"


await run_tests([
    test_rate_limit_single_domain,
    test_rate_limit_multiple_domains,
    test_robots_parsing,
    test_blocked_urls,
    test_delays_respected,
    test_domains_in_parallel,
])

✅ test_rate_limit_single_domain


19:09:12 [WARNING] Заблокировано robots.txt: https://polite.com/private/secret


✅ test_rate_limit_multiple_domains
✅ test_robots_parsing
Обработано: 2 | В очереди: 0 | Ошибок: 0 | Заблокировано: 1 | Скорость: 19.6 стр/сек | Запросов: 2.0 req/sec | Средняя задержка: 0.00 сек
✅ test_blocked_urls
✅ test_delays_respected
Обработано: 4 | В очереди: 10 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.0 стр/сек | Запросов: 4.0 req/sec | Средняя задержка: 0.50 сек
Обработано: 8 | В очереди: 6 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.0 стр/сек | Запросов: 4.0 req/sec | Средняя задержка: 0.50 сек
Обработано: 12 | В очереди: 2 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.0 стр/сек | Запросов: 4.0 req/sec | Средняя задержка: 0.50 сек
Обработано: 14 | В очереди: 0 | Ошибок: 0 | Заблокировано: 0 | Скорость: 4.5 стр/сек | Запросов: 4.5 req/sec | Средняя задержка: 0.50 сек
✅ test_domains_in_parallel

Пройдено 6 из 6
